# 22. Streamlit Data Apps — `streamlit`
**Python Libraries Lab · Phase 2 · Data Visualisation**

_Turn a Python script into a web app, no HTML needed._

⚙️ **Runs in Colab** after the setup cell below.

Streamlit turns a normal Python script into an interactive web app. You write st dot title, st dot slider, st dot dataframe, and Streamlit builds the page. When someone moves a slider, Streamlit simply reruns your script from top to bottom with the new value. That simple model is why people build dashboards in an afternoon.

**In this notebook:** You will build a ShopEasy sales dashboard with a city filter, a category picker, live summary numbers, a chart and a table, and then test it automatically.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q streamlit
import importlib.metadata as md
for pkg in ['streamlit']:
    print(pkg, md.version(pkg))

## Step 1 · Write the app file
`%%writefile` saves the cell as `app.py`. Read it top to bottom: that is exactly what runs on every click.

In [ ]:
%%writefile app.py
import numpy as np, pandas as pd
import streamlit as st

st.set_page_config(page_title="ShopEasy sales", layout="wide")

@st.cache_data
def load():
    rng = np.random.default_rng(42)
    n = 600
    df = pd.DataFrame({
        "date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
        "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
        "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n),
        "revenue": rng.integers(199, 9000, n),
    })
    return df

df = load()
st.title("ShopEasy sales dashboard")
city = st.selectbox("City", ["All"] + sorted(df["city"].unique()))
cats = st.multiselect("Categories", sorted(df["category"].unique()), default=sorted(df["category"].unique()))

view = df[df["category"].isin(cats)]
if city != "All":
    view = view[view["city"] == city]

c1, c2, c3 = st.columns(3)
c1.metric("Orders", f"{len(view):,}")
c2.metric("Revenue", f"Rs.{view['revenue'].sum():,}")
c3.metric("Average order", f"Rs.{view['revenue'].mean():,.0f}" if len(view) else "-")

st.line_chart(view.groupby("date")["revenue"].sum())
st.dataframe(view.sort_values("revenue", ascending=False).head(20), width="stretch")

## Step 2 · Test the app without a browser
`AppTest` runs the script, lets you set widgets, and checks what the page shows.

In [ ]:
import os
from streamlit.testing.v1 import AppTest

APP = os.path.abspath("app.py")          # AppTest needs the full path when used in a notebook
import logging
logging.getLogger("streamlit").setLevel(logging.ERROR)   # hide harmless "bare mode" warnings
at = AppTest.from_file(APP).run(timeout=30)
print("Title:", at.title[0].value)
print("Metrics:", [(m.label, m.value) for m in at.metric])

at.selectbox[0].select("Pune").run()
print("After choosing Pune:", [(m.label, m.value) for m in at.metric])
assert not at.exception, at.exception

## Step 3 · Start the server
Runs Streamlit in the background on port 8501. Wait about 5 seconds for it to start.

In [ ]:
import subprocess, time
server = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                          stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
print("\n".join(l for l in open("streamlit.log").read().splitlines() if "Local URL" in l) or open("streamlit.log").read()[-300:])

## Step 4 · Open the app
In Colab this opens the live app in a window below the cell. On your laptop, open the Local URL printed above.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8501, path="/")
except ImportError:
    print("Not in Colab: open http://localhost:8501 in your browser")

## Step 5 · Stop the server
Always stop background servers when you are done.

In [ ]:
server.terminate()
print("Server stopped")

## ✍️ Your turn
Use `AppTest` to choose **only the 'Books' category** (multiselect) and store the **Orders metric value** (a string like `'150'`) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # hint: at.multiselect[0].set_value([...]).run()

In [ ]:
def check():
    t = AppTest.from_file(APP).run(timeout=30)
    t.multiselect[0].set_value(["Books"]).run()
    expected = t.metric[0].value
    return answer == expected, f"Expected {expected!r}, got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
at = AppTest.from_file(APP).run(timeout=30)
at.multiselect[0].set_value(["Books"]).run()
answer = at.metric[0].value

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **st.write**
- **Widgets**
- **Rerun model**
- **st.cache_data**
- **AppTest**

**Next:** 23. scikit-learn (`scikit-learn`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/23_scikit_learn.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install streamlit
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.